# Meetod 2 — CSV + ühine DataFrame

**Koolitusmaterjalides kirjeldatud CSV-alternatiiv.**

Andmevoog:

`CSV → df_sales + df_customers → merge → df → puhastus → RFM → visualiseerimine`

Meetod 1-ga võrreldes muutub ainult andmete laadimine. Pärast `df_sales` ja `df_customers` loomist on merge, puhastus, RFM ja visualiseerimine sama.

**Õppekatse täpsustus:** koolituse ametlik varuplaan viitab puhastatud CSV-dele. Selles isiklikus katses kasutame teadlikult toor-CSV-sid, et harjutada ka Roll B puhastamist.


## Roll A — CSV-failidest laadimine

**Mida teeme?** Loeme kaks toor-CSV faili pandas DataFrame'idesse.

**Miks?** `pd.read_csv()` on Week 7 materjalides näidatud alternatiiv Supabase'ile. Kui DataFrame'ide struktuur on sama, ei pea järgmised analüüsietapid muutuma.

**Kontroll:** vaatame `shape` ja `head()`, et veenduda failide sisus enne merge'i.


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df_sales = pd.read_csv("../data/raw/sales.csv")
df_customers = pd.read_csv("../data/raw/customers.csv")

print("Sales:", df_sales.shape)
display(df_sales.head())

print("Customers:", df_customers.shape)
display(df_customers.head())


Sales: (15234, 11)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


Customers: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004


### Roll A — tabelite ühendamine

**Mida teeme?** Ühendame müügid kliendiandmetega `customer_id` järgi.

**Miks?** `LEFT` merge säilitab kõik müügiread. Kliendiandmed lisatakse ainult siis, kui vaste on olemas.

**Väljund:** ühendatud DataFrame `df`, mille Roll B saab puhastamiseks.


In [2]:
customer_columns = ["customer_id", "first_name", "last_name", "email", "phone", "city"]

df = pd.merge(
    df_sales,
    df_customers[customer_columns],
    on="customer_id",
    how="left"
)

print("Liidetud tabeli shape:", df.shape)
print(df.dtypes)
display(df.head())


Liidetud tabeli shape: (15234, 16)
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
first_name            str
last_name             str
email                 str
phone                 str
city                  str
dtype: object


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu


### Roll A järeldused
- enne merge'i sales: 15 234 rida  vs pärast merge'i: 15 234 rida  > Seega LEFT merge ei kaotanud ega paljundanud müügiridu.
- Roll A tööosas `sale_date` tuli CSV-st tekstina → Roll B peab selle muutma datetime-iks.
- `customer_id` tuli float64-na, sest selles väljas on toorandmetes puuduvaid väärtusi (NaN). See on pandas'e CSV-lugemise normaalne tagajärg

## Roll B — andmete puhastamine

**Mida teeme?** Puhastame Roll A väljundi enne kliendipõhist RFM-analüüsi.

**Miks?**
- korduv `invoice_id` võib sama müügi topelt arvesse võtta;
- puuduva `customer_id`, `sale_date` või `total_price` väärtusega rida ei sobi RFM-i;
- `sale_date` peab enne kuupäevaarvutusi olema datetime;
- `total_price <= 0` ei lähe positiivse ostukäibe RFM-analüüsi;
- juhendi viitekuupäevast hilisemad müügid tuleb enne RFM-i eemaldada, muidu võib Recency muutuda negatiivseks.

**Väljund:** puhastatud `df`, mida Roll C kasutab.


In [3]:
print("Esialgne shape:", df.shape)

# Kontrollime mõju enne eemaldamist, et puhastus oleks nähtav.
print("Duplikaate invoice_id järgi:", df.duplicated(subset=["invoice_id"]).sum())
print("NULL väärtused RFM-väljadel:")
print(df[["customer_id", "sale_date", "total_price"]].isna().sum())

df = df.drop_duplicates(subset=["invoice_id"], keep="first")
df = df.dropna(subset=["customer_id", "sale_date", "total_price"])

df["sale_date"] = pd.to_datetime(
    df["sale_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Vigaseid kuupäevi pärast teisendamist:", df["sale_date"].isna().sum())
df = df.dropna(subset=["sale_date"]).copy()

print("Mittepositiivseid müügisummasid:", (df["total_price"] <= 0).sum())
df = df[df["total_price"] > 0].copy()

today = pd.to_datetime("2025-02-28")
print("Viitekuupäevast hilisemaid müügiridu:", (df["sale_date"] > today).sum())
df = df[df["sale_date"] <= today].copy()

print("\nPuhastatud shape:", df.shape)
print("Unikaalseid kliente:", df["customer_id"].nunique())
print("Kuupäevavahemik:", df["sale_date"].min(), "kuni", df["sale_date"].max())


Esialgne shape: (15234, 16)
Duplikaate invoice_id järgi: 5116
NULL väärtused RFM-väljadel:
customer_id    1487
sale_date         0
total_price       0
dtype: int64
Vigaseid kuupäevi pärast teisendamist: 0
Mittepositiivseid müügisummasid: 180
Viitekuupäevast hilisemaid müügiridu: 238

Puhastatud shape: (8712, 16)
Unikaalseid kliente: 2515
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2025-02-28 00:00:00


## Roll C — RFM kliendisegmenteerimine

**Mida teeme?** Arvutame iga kliendi Recency, Frequency ja Monetary väärtused.

**Miks?**
- **Recency** näitab, kui hiljuti klient ostis;
- **Frequency** näitab ostude arvu;
- **Monetary** näitab analüüsi kaasatud ostude koguväärtust.

Järgime Week 7 grupijuhendi baastaset: Frequency arvutatakse `sale_id.count()` abil. Kõigis kolmes meetodis kasutatakse sama RFM-loogikat.

Roll B on juba eemaldanud `2025-02-28` hilisemad müügid. Roll C määrab sama `today` väärtuse Recency arvutuse jaoks.


In [4]:
today = pd.to_datetime("2025-02-28")

recency = df.groupby("customer_id")["sale_date"].max().reset_index()
recency.columns = ["customer_id", "last_purchase_date"]
recency["recency_days"] = (today - recency["last_purchase_date"]).dt.days

frequency = df.groupby("customer_id")["sale_id"].count().reset_index()
frequency.columns = ["customer_id", "frequency"]

monetary = df.groupby("customer_id")["total_price"].sum().reset_index()
monetary.columns = ["customer_id", "monetary_value"]

rfm = (
    recency[["customer_id", "recency_days"]]
    .merge(frequency, on="customer_id")
    .merge(monetary, on="customer_id")
)

print("RFM shape:", rfm.shape)
display(rfm.head())


RFM shape: (2515, 4)


,customer_id,recency_days,frequency,monetary_value
0,2001.0,91,2,203.92
1,2004.0,71,2,1198.56
2,2005.0,164,4,959.60
3,2006.0,536,1,327.06
4,2007.0,29,1,318.63


### RFM kvaliteedikontroll — Recency
Kontrollime enne RFM-skooride arvutamist, et Recency väärtused oleksid kooskõlas valitud viitekuupäevaga.

Negatiivsete Recency väärtuste puudumine kinnitab, et viitekuupäevast `2025-02-28` hilisemad müügid on analüüsist korrektselt välja jäetud.

In [5]:
print("Recency min:", rfm["recency_days"].min())
print("Recency max:", rfm["recency_days"].max())
print("Negatiivseid Recency väärtusi:", (rfm["recency_days"] < 0).sum())

Recency min: 0
Recency max: 778
Negatiivseid Recency väärtusi: 0


### RFM-skoorid

**Mida teeme?** Jagame kliendid iga RFM-mõõdiku järgi viide gruppi ja anname skoori 1–5.

**Miks on Recency vastupidine?** Väiksem päevade arv tähendab värskemat ostu ja seetõttu kõrgemat skoori.


In [6]:
rfm["R_score"] = pd.qcut(rfm["recency_days"], 5, labels=[5, 4, 3, 2, 1])
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
rfm["M_score"] = pd.qcut(rfm["monetary_value"], 5, labels=[1, 2, 3, 4, 5])

rfm[["R_score", "F_score", "M_score"]] = rfm[["R_score", "F_score", "M_score"]].astype(int)
rfm["RFM_Score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

display(rfm.head())


,customer_id,recency_days,frequency,monetary_value,R_score,F_score,M_score,RFM_Score
0,2001.0,91,2,203.92,4,1,1,6
1,2004.0,71,2,1198.56,5,1,4,10
2,2005.0,164,4,959.60,3,4,4,11
3,2006.0,536,1,327.06,1,1,1,3
4,2007.0,29,1,318.63,5,1,1,7


### RFM kvaliteedikontroll — skoorid

Kontrollime enne klientide segmentimist, et R-, F- ja M-skoorid jääksid oodatud vahemikku 1–5 ning RFM-koondskoor vahemikku 3–15.

In [7]:
print("R_score:", rfm["R_score"].min(), "-", rfm["R_score"].max())
print("F_score:", rfm["F_score"].min(), "-", rfm["F_score"].max())
print("M_score:", rfm["M_score"].min(), "-", rfm["M_score"].max())
print("RFM_Score:", rfm["RFM_Score"].min(), "-", rfm["RFM_Score"].max())

R_score: 1 - 5
F_score: 1 - 5
M_score: 1 - 5
RFM_Score: 3 - 15


### Kliendisegmendid

**Mida teeme?** Tõlgime RFM-koondskoori äriliselt arusaadavateks segmentideks.

**Kontroll:** pärast segmentimist ei tohi ükski klient jääda segmendita ja osakaalud peavad kokku andma ligikaudu 100%.


In [8]:
def segment_customer(row):
    if row["RFM_Score"] >= 13:
        return "VIP Champions"
    elif row["RFM_Score"] >= 10:
        return "Loyal"
    elif row["RFM_Score"] >= 7:
        return "Potential"
    elif row["RFM_Score"] >= 4:
        return "At Risk"
    return "Lost"


rfm["Segment"] = rfm.apply(segment_customer, axis=1)

segment_summary = rfm["Segment"].value_counts().rename_axis("Segment").reset_index(name="customers")
segment_summary["customer_share_pct"] = (segment_summary["customers"] / len(rfm) * 100).round(2)

display(segment_summary)
print("Segmendita kliente:", rfm["Segment"].isna().sum())
print("Osakaal kokku:", round(segment_summary["customer_share_pct"].sum(), 2), "%")


,Segment,customers,customer_share_pct
0,Potential,740,29.42
1,Loyal,684,27.20
2,At Risk,512,20.36
3,VIP Champions,455,18.09
4,Lost,124,4.93


Segmendita kliente: 0
Osakaal kokku: 100.0 %


## Roll D — visualiseerimine

**Mida teeme?** Loome juhendis nõutud kolm Plotly visualiseeringut.

**Miks?** Iga graafik vastab eraldi äriküsimusele: segmentide suurus, kliendi värskuse ja väärtuse seos ning kõige väärtuslikumad VIP-kliendid.


In [9]:
segment_counts = rfm["Segment"].value_counts().reset_index()
segment_counts.columns = ["Segment", "Klientide arv"]

fig_segments = px.bar(
    segment_counts,
    x="Segment",
    y="Klientide arv",
    title="Klientide jaotus RFM-segmentide kaupa",
    text="Klientide arv",
    color_discrete_sequence=["#009B8D"]
)

fig_segments.show()


In [10]:
segment_colors = {
    "VIP Champions": "#008577",  # UrbanStyle teal
    "Loyal": "#1A1A2E",          # tume navy
    "Potential": "#2F6690",      # selge sinine
    "At Risk": "#D97706",        # tumedam oranž
    "Lost": "#C1121F"            # punane
}

fig_scatter = px.scatter(
    rfm,
    x="recency_days",
    y="monetary_value",
    color="Segment",
    size="frequency",
    hover_data=["customer_id"],
    color_discrete_map=segment_colors,
    category_orders={
        "Segment": [
            "VIP Champions",
            "Loyal",
            "Potential",
            "At Risk",
            "Lost"
        ]
    },
    log_y=True,
    title="UrbanStyle kliendisegmendid (RFM)",
    labels={
        "recency_days": "Päevi viimasest ostust",
        "monetary_value": "Kogukulutus (EUR)"
    }
)

fig_scatter.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig_scatter.show()

In [ ]:
# TOP 10 VIP klienti koos nimedega
top_10_vip = (
    rfm[rfm["Segment"] == "VIP Champions"]
    .nlargest(10, "monetary_value")
    .merge(
        df[["customer_id", "first_name", "last_name"]].drop_duplicates("customer_id"),
        on="customer_id",
        how="left"
    )
)

top_10_vip["customer_name"] = top_10_vip["first_name"] + " " + top_10_vip["last_name"]

# Keskmised
avg_all = rfm["monetary_value"].mean()
avg_vip = rfm.loc[rfm["Segment"] == "VIP Champions", "monetary_value"].mean()

# Graafik
fig_top_vip = px.bar(
    top_10_vip,
    x="customer_name",
    y="monetary_value",
    text_auto=".2f",
    title="TOP 10 VIP klienti kogukulutuse järgi",
    labels={"customer_name": "Klient", "monetary_value": "Kogukulutus (EUR)"},
    color_discrete_sequence=["#008577"]
)

# Tulpade väärtused horisontaalselt
fig_top_vip.update_traces(textangle=0)

# Võrdlusjooned
fig_top_vip.add_trace(go.Scatter(
    x=top_10_vip["customer_name"],
    y=[avg_all] * 10,
    mode="lines",
    name=f"Kõigi klientide keskmine: {avg_all:.2f} EUR",
    line=dict(color="#7F7F7F", dash="dash")
))

fig_top_vip.add_trace(go.Scatter(
    x=top_10_vip["customer_name"],
    y=[avg_vip] * 10,
    mode="lines",
    name=f"VIP klientide keskmine: {avg_vip:.2f} EUR",
    line=dict(color="#A6A6A6", dash="dot")
))

# Kujundus
fig_top_vip.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=115),
    legend=dict(x=1, y=1.16, xanchor="right", yanchor="top")
)

fig_top_vip.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.Scatter: 'textangle'

Did you mean "text"?

    Valid properties:
        alignmentgroup
            Set several traces linked to the same position axis or
            matching axes to the same alignmentgroup. This controls
            whether bars compute their positional range dependently
            or independently.
        cliponaxis
            Determines whether or not markers and text nodes are
            clipped about the subplot axes. To show markers and
            text nodes above axis lines and tick labels, make sure
            to set `xaxis.layer` and `yaxis.layer` to *below
            traces*.
        connectgaps
            Determines whether or not gaps (i.e. {nan} or missing
            values) in the provided data arrays are connected.
        customdata
            Assigns extra data each datum. This may be useful when
            listening to hover, click and selection events. Note
     

## Lõppkontroll

Meetod 2 peab töötama `Restart Kernel → Run All` abil.

Kui Meetod 1 ja Meetod 2 saavad sisuliselt sama algandmestiku ning rakendavad samu puhastusreegleid, peaksid RFM-tulemused kokku langema. Kui ei lange, kontrolli esmalt sisendi ridade arvu, kuupäevi ja puhastuse mõju.
